# Abstract Base Classes (ABCs) – Practice Exercises

This notebook contains hands-on exercises to practice Python's **Abstract Base Classes (ABCs)** using the `abc` module.

For each exercise:

- Read the description carefully.
- Implement your solution in the provided code cell (in the `# TODO` block).
- Add small tests at the bottom of the cell to verify behavior.


### Contents

- [Exercise 1 – Basic Abstract Method](#exercise-1--basic-abstract-method)
- [Exercise 2 – Abstract Property](#exercise-2--abstract-property)
- [Exercise 3 – Abstract Method with Arguments](#exercise-3--abstract-method-with-arguments)
- [Exercise 4 – Abstract Classmethod as a Constructor](#exercise-4--abstract-classmethod-as-a-constructor)
- [Exercise 5 – Template Method with an Abstract Hook](#exercise-5--template-method-with-an-abstract-hook)
- [Exercise 6 – Register Virtual Subclasses](#exercise-6--register-virtual-subclasses)
- [Exercise 7 – Mixins + Multiple Inheritance](#exercise-7--mixins--multiple-inheritance)
- [Exercise 8 – Trading Case Study: Order + Risk Model](#exercise-8--trading-case-study-order--risk-model)
- [Exercise 9 – Enforce Interface via `__subclasshook__`](#exercise-9--enforce-interface-via-__subclasshook__)


## Exercise 1 – Basic Abstract Method

**Goal**: Define an ABC with a single abstract method, then implement a concrete subclass.

**Requirements**:

- Create `BaseAction(ABC)`.
- Add `@abstractmethod def run(self) -> str:`.
- Implement `class BuyAction(BaseAction)` with `run()` returning a string.
- Confirm `BaseAction()` cannot be instantiated (expect `TypeError`).

**Hint**: If something is abstract, instantiation fails at runtime.


In [ ]:
# Exercise 1 – implement BaseAction and BuyAction

from abc import ABC, abstractmethod

class BaseAction(ABC):
    @abstractmethod
    def run(self): ...

class BuyAction(BaseAction):
    def run(self): return f"...running {self.__class__.__name__}..."

try: print(BaseAction().run())
except Exception as EXC:
    print(repr(EXC))

try: print(BuyAction().run())
except Exception as EXC:
    print(repr(EXC))

TypeError("Can't instantiate abstract class BaseAction without an implementation for abstract method 'run'")
...running BuyAction...


## Exercise 2 – Abstract Property

**Goal**: Use `@property` together with `@abstractmethod`.

**Requirements**:

- Create `BaseInstrument(ABC)` with an abstract `symbol` property (read-only).
- Implement `EquityInstrument(BaseInstrument)`.
- Show that accessing `.symbol` works.

**Hint**: Use `@property` and decorate it with `@abstractmethod`.


In [20]:
# Exercise 2 – implement BaseInstrument and EquityInstrument

from abc import ABC, abstractmethod
from typing import AnyStr

class BaseInstrument(ABC):
    def __init__(self, symbol: str):
        self._symbol = symbol
    @abstractmethod
    def symbol(self): ...

class EquityInstrument(BaseInstrument):
    @property
    def symbol(self): return self._symbol

ins = EquityInstrument("MSFT")
print("Symbol name:", ins.symbol)
try: ins.symbol = "TFSM"
except Exception as EXC:
    print(repr(EXC))

Symbol name: MSFT
AttributeError("property 'symbol' of 'EquityInstrument' object has no setter")


## Exercise 3 – Abstract Method with Arguments

**Goal**: Define an abstract method that takes arguments, then implement it.

**Requirements**:

- Create `BasePricer(ABC)` with `@abstractmethod def price(self, qty: float, limit: float | None) -> float:`.
- Implement `SimplePricer(BasePricer)`.
- In `price()`, implement a rule like: if `limit` is not None, return `min(qty, limit)`; otherwise return `qty`.

**Hint**: Keep your implementation simple so the focus stays on ABC mechanics.


In [ ]:
# Exercise 3 – implement BasePricer and SimplePricer

from abc import ABC, abstractmethod
class BasePricer(ABC):
    @abstractmethod
    def price(self, qty: float, limit: float = None): ...

class SimplePricer(BasePricer):
    def price(self, qty: float, limit: float = None):
        if limit is not None: return min(qty, limit)
        else: return qty

sp = SimplePricer()
sp.price(10, 9)

9

## Exercise 4 – Abstract Classmethod as a Constructor

**Goal**: Make the ABC require a `@classmethod` that constructs an instance from configuration.

**Requirements**:

- Create `Configurable(ABC)` with an abstract `@classmethod def from_config(cls, cfg: dict) -> 'Configurable':`.
- Implement `RiskModel(Configurable)`.
- `from_config` should read `cfg['max_loss']` and store it.

**Hint**: Use quotes in the return type if needed.


In [29]:
# Exercise 4 – implement Configurable and RiskModel

from abc import ABC, abstractmethod
from typing import Any, Dict

class Configurable(ABC):
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)
    @classmethod
    @abstractmethod
    def from_config(cls, cfg: Dict[str, Any]): ...

class RiskModel(Configurable):
    @classmethod
    def from_config(cls, cfg):
        return cls(max_loss = cfg["max_loss"])

cfg = {"max_loss": 0.12}
rm = RiskModel.from_config(cfg)

assert rm.max_loss == cfg["max_loss"]
print("Max loss stored:", rm.max_loss)

Max loss stored: 0.12


## Exercise 5 – Template Method with an Abstract Hook

**Goal**: Implement the Template Method pattern with one abstract hook method.

**Requirements**:

- Create `BaseOrder(ABC)`.
- Provide a concrete method `execute(broker)` that calls:
          - `self._build_payload()` (abstract hook)
          - then `broker.send(payload)`.
- Implement `MarketOrder(BaseOrder)` with a concrete `_build_payload`.
- Create a tiny `Broker` class with `send()` that prints the payload.

**Hint**: Concrete `execute()` should be the same for all order types.


In [ ]:
# Exercise 5 – implement BaseOrder and MarketOrder

from abc import ABC, abstractmethod
from typing import Dict, Any

class Broker:
    def __init__(self, name: str, url: str):
        self.name, self.url = name, url
    def send(self, payload: Dict[str, Any]):
        print(f"\"{self.name}\" sending => {payload}")

class BaseOrder(ABC):
    def execute(self, broker: Broker):
        payload = self._build_payload()
        broker.send(payload)
    @abstractmethod
    def _build_payload(self): ...

class MarketOrder(BaseOrder):
    def __init__(self, *args, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)
    def _build_payload(self):
        return self.__dict__.copy()

broker = Broker("IBKR", "https://www.ibkr.com/")
order = MarketOrder(size = 1, side = "Buy")
order.execute(broker)

"IBKR" sending => {'size': 1, 'side': 'Buy'}


## Exercise 6 – Register Virtual Subclasses

**Goal**: Use `ABC.register()` so `isinstance()` works without inheriting.

**Requirements**:

- Create `class Payment(ABC)` with an abstract method `pay(self, amount: float) -> None`.
- Create a *separate* class `LegacyPayer` that has a compatible `pay()` method but does not inherit from `Payment`.
- Register it using `Payment.register(LegacyPayer)`.
- Show that `isinstance(LegacyPayer(), Payment)` becomes True.

**Hint**: Registration affects `isinstance`/`issubclass` checks.


In [36]:
# Exercise 6 – implement Payment + virtual subclass registration

from abc import ABC, abstractmethod

class Payment(ABC):
    @abstractmethod
    def pay(self, amount: float): ...
class LegacyPayer:
    def pay(self, amount: float):
        print(f"Amount paid: {amount:.2f}")

Payment.register(LegacyPayer)
line = f"\"{LegacyPayer.__name__}\" as {{definition}} of {Payment.__name__}:"
print(line.format(definition = "instance"), isinstance(LegacyPayer(), Payment))
print(line.format(definition = "subclass"), issubclass(LegacyPayer, Payment))

"LegacyPayer" as instance of Payment: True
"LegacyPayer" as subclass of Payment: True


## Exercise 7 – Mixins + Multiple Inheritance

**Goal**: Combine an ABC contract with a reusable mixin.

**Requirements**:

- Create `class LogMixin:` with a concrete `log(self, msg: str) -> None`.
- Create `class Runner(ABC)` with an abstract `run()`.
- Create `class LoggingRunner(LogMixin, Runner)` that implements `run()` and uses `self.log()`.

**Hint**: Multiple inheritance order matters for method lookup.


In [53]:
# Exercise 7 – implement LogMixin + Runner + LoggingRunner

from abc import ABC, abstractmethod
from typing import Any, Dict
from datetime import datetime, timedelta, timezone

class LogMixin:
    tz = timezone(timedelta(0))
    def __init__(self, log_prefix: str = "LOG", **kwargs):
        super().__init__(**kwargs)
        self.log_prefix = log_prefix
    def log(self, msg: str):
        now = datetime.now(tz = self.tz)
        now_str = now.strftime("%H:%M:%S.%f")
        print(f"[{now_str[:-3]}|{self.log_prefix}]", msg)

class Runner(ABC):
    def __init__(self, **kwargs):
        super().__init__()
        self._active = False
        for items in kwargs.items():
            setattr(self, *items)
    @abstractmethod
    def run(self): ...

class LoggingRunner(LogMixin, Runner):
    def __init__(self, log_prefix: str = "LR", *args, **kwargs):
        super().__init__(log_prefix = log_prefix, *args, **kwargs)
    def run(self):
        self._active = True
        self.log(f"Running ({self._active = })")

lr = LoggingRunner(duration = timedelta(days = 2), log_prefix = "RNNR")
lr.run()

[02:11:36.849|RNNR] Running (self._active = True)


## Exercise 8 – Trading Case Study: Order + Risk Model

**Goal**: Apply ABCs to a small trading-like scenario with a contract for risk checking.

**Requirements**:

- Create `class BaseOrder(ABC)` with `execute()` (concrete) that calls `validate(risk_model)` then `broker.send()`.
- Create `class BaseRiskModel(ABC)` with abstract `check(order) -> bool`.
- Implement:
- `LimitOrder(symbol, qty, limit)`
- `MaxQtyRiskModel(max_qty)`
- Demo: validate should return False if `abs(order.qty) > max_qty`.

**Hint**: Keep the objects small; the point is ABC structure.


In [ ]:
# Exercise 8 – implement BaseOrder + BaseRiskModel + LimitOrder + MaxQtyRiskModel

from abc import ABC, abstractmethod
from dataclasses import dataclass

class Broker:
    def __init__(self, name: str, url: str):
        self.name, self.url = name, url
    def send(self, payload: Dict[str, Any]):
        print(f"\"{self.name}\" sending => {payload}")

class BaseRiskModel(ABC):
    def __init__(self, **kwargs):
        for item in kwargs.items():
            setattr(self, *item)
    @abstractmethod
    def check(self, order: BaseOrder): ...

class BaseOrder(ABC):
    def __init__(self, qty: float, **kwargs):
        self.qty = qty
        for item in kwargs.items(): setattr(self, *item)
    def execute(self, broker: Broker, risk_model: BaseRiskModel):
        assert self.validate(risk_model)
        broker.send(self._build_payload())
    def validate(self, risk_model: BaseRiskModel):
        return risk_model.check(self)
    def _build_payload(self):
        return {"qty": self.qty}

class MaxQtyRiskModel(BaseRiskModel):
    def __init__(self, max_qty: float, crt_qty: float, **kwargs):
        self.max_qty, self.crt_qty = max_qty, crt_qty
        super().__init__(**kwargs)
    def check(self, order: BaseOrder):
        if (order.qty < self.crt_qty):
            order.qty = min(order.qty, self.max_qty)
            return True
        else: 
            print(f"Order with quantity ({order.qty}) above critical ({self.crt_qty})")
            return False
        
class LimitOrder(BaseOrder):
    def __init__(self, qty: float, side: str, price: float, **kwargs):
        self.side, self.price = side, price
        super().__init__(qty, **kwargs)

broker = Broker("IBKR", "https://www.ibkr.com/")
rmodel = MaxQtyRiskModel(0.1, 0.2)
order = LimitOrder(0.05, "buy", 1.23456)
order.execute(broker, rmodel)

"IBKR" sending => {'qty': 0.05}


## Exercise 9 – Enforce Interface via `__subclasshook__`

**Goal**: Implement structural checks using `__subclasshook__`.

**Requirements**:

- Create an ABC `HasExecute(ABC)`.
- Implement `__subclasshook__` so that any class with a callable `execute` attribute is considered a virtual subclass.
- Test with two classes:
- `Good` defines `execute()`
- `Bad` does not.
- Verify `issubclass(Good, HasExecute)` and `issubclass(Bad, HasExecute)`.

**Hint**: In `__subclasshook__`, return `True/False` and handle `NotImplemented` appropriately.


In [96]:
# Exercise 9 – implement HasExecute.__subclasshook__

from abc import ABC, abstractmethod
from typing import Callable, ClassVar

class HasExecute(ABC):
    @classmethod
    def __subclasshook__(cls: ClassVar, scls: ClassVar):
        for base in scls.__mro__:
            execute: Callable = base.__dict__.get("execute", None)
            if not callable(execute): continue
            is_execute_abstract = getattr(execute, "__isabstractmethod__", False)
            if is_execute_abstract: continue
            return True
        return False

    @abstractmethod
    def execute(self): pass

class GoodExecute(HasExecute):
    def execute(self):
        print("Executed")

class BadExecute(HasExecute):
    def executet(self):
        print("Executeted")

try: ge = GoodExecute() ; ge.execute()
except Exception as EXC:
    print("GoodExecute error:", repr(EXC))
try: be = BadExecute() ; be.execute()
except Exception as EXC:
    print("BadExecute error:", repr(EXC))

verbose = "Is \"{}\" subclass of \"{}\": \"{}\""
print(verbose.format(GoodExecute.__name__, HasExecute.__name__,
      issubclass(GoodExecute, HasExecute)))
print(verbose.format(BadExecute.__name__, HasExecute.__name__,
      issubclass(BadExecute, HasExecute)))

Executed
BadExecute error: TypeError("Can't instantiate abstract class BadExecute without an implementation for abstract method 'execute'")
Is "GoodExecute" subclass of "HasExecute": "True"
Is "BadExecute" subclass of "HasExecute": "False"
